In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import pickle
import numpy as np
import json

from fpp.utils.validation import pp_finite_sample_band
from fpp.utils.posterior import multi_corner

import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc_file("../src/fpp/utils/matplotlibrc")

/n/home07/yitians/.conda/envs/torch/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


# TMP

In [ ]:
from fpp.models.np_model import NPModel
m = NPModel()
z = np.load(os.environ['MYSTORE'] + f"/fermi/fermi-prob-prog/outputs/production/simulations/fullprior42.npy")

No data provided. Using Fermi data.


In [9]:
for i in range(100):
    data = z[i]
    print(f'{i}, {np.max(data[~m.mask_roi]):5.0f}, {np.max(data[~m.nm]):5.0f}, {np.max(data):5.0f}')

0,   246,  1349,  1349
1,   130, 13863, 13863
2,   187, 11207, 11207
3,   159, 16028, 16028
4,   283,  4212,  4212
5,   257, 14423, 14423
6,   143,  8446,  8446
7,   179, 12387, 12387
8,   205, 12216, 12216
9,   160,  4322,  4322
10,   121, 12500, 12500
11,   176,  2857,  2857
12,   176,  2921,  2921
13,   122, 14334, 14334
14,   124, 13117, 13117
15,   148,  6844,  6844
16,   146,  9126,  9126
17,   167,  8921,  8921
18,   228, 14791, 14791
19,   148,  3246,  3246
20,   154,  1506,  1506
21,   149, 10410, 10410
22,   153,  2973,  2973
23,   183,   918,   918
24,   110,  9963,  9963
25,   138,  8257,  8257
26,   139, 16134, 16134
27,   120,  7308,  7308
28,   150, 14757, 14757
29,   225,  4047,  4047
30,   213,  5181,  5181
31,   222,  1751,  1751
32,   185,  3772,  3772
33,   251, 11821, 11821
34,   215,  4067,  4067
35,   149,  5830,  5830
36,   172, 11232, 11232
37,   188, 10656, 10656
38,   323,   386,   386
39,   186,   565,   565
40,   314,  7608,  7608
41,   124, 10075, 10075
42

# Coverage

In [11]:
run_name = 'calibration/hmctd10-old-delta-2'
z = pickle.load(open(os.environ['MYSTORE'] + f'/fermi/fermi-prob-prog/outputs/production/fits/{run_name}/p_nominal_actual_dict.p', 'rb'))
print(' '.join(z.keys()))

S_bub S_gce S_ics S_iso S_pib S_psc Sps_dsk n1_dsk n2_dsk n3_dsk sb1_dsk lambdas_dsk Sps_gce n1_gce n2_gce n3_gce sb1_gce lambdas_gce f_bulge_poiss f_bulge_ps gamma_poiss gamma_ps C zs


In [ ]:
if 'pois' in run_name:
    labels = [
        'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce',
        'f_bulge_poiss', 'gamma_poiss',
    ]
else:
    labels = [
        'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce',
        'Sps_dsk', 'Sps_gce',
        'f_bulge_poiss', 'f_bulge_ps', 'gamma_poiss', 'gamma_ps', 'C', 'zs'
    ]

probs = [z[k] for k in labels]
ls_s = ['-'] * 10 + [':'] * 10

fig, ax = plt.subplots()

ax.fill_between([0,1], [0,1], color='lightgray')
if labels is None:
    labels = [None for _ in probs]
for prob, label, ls in zip(probs, labels, ls_s):
    ax.plot(prob[0], prob[1], label=label, ls=ls)

n_run = len(probs[0][0])
lower, upper = pp_finite_sample_band(n_run)
ax.plot(upper, np.linspace(0, 1, n_run), 'k:', label=f'{n_run} sample 95\% \ncontainment')
ax.plot(lower, np.linspace(0, 1, n_run), 'k:')

ax.set(aspect=1, xlim=(0, 1), ylim=(0, 1))
ax.set(xlabel='Nominal coverage', ylabel='Actual coverage', title=run_name)

fig.legend(bbox_to_anchor=(1, 1), loc='upper left', bbox_transform=ax.transAxes)
plt.tight_layout()

# Corner

In [ ]:
point_est = True
truth_dict = json.load(open("../outputs/truths/truth_dict_base230927.json", 'r'))
labels = [
    'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce', 'Sps_dsk', 'Sps_gce',
    'f_bulge_poiss', 'f_bulge_ps', 'gamma_poiss', 'gamma_ps', 'C', 'zs'
]
config_dict = {
    'hmc' : ('/n/home07/yitians/fermi/fermi-prob-prog/outputs/production/fits/calibration/hmc-old-delta-2/4.p', 'k'),
    'pt' : ('/n/home07/yitians/fermi/fermi-prob-prog/outputs/production/fits/calibration/pthmc-old-delta-2/4.p', 'r'),
}

s_in = {}
labels_dict = {}
colors_dict = {}
for key, (path, color) in config_dict.items():
    s = pickle.load(open(path, 'rb'))
    s_in[key] = {k: s[k] for k in labels}
    labels_dict[key] = key
    colors_dict[key] = color

t_in = {k: truth_dict[k] for k in labels} if point_est else None

multi_corner(s_in, labels, point_est=t_in, colors_dict=colors_dict, labels_dict=labels_dict)